# 2장 2강 경사하강법, 옵티마이저 및 데이터 분할

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('C:/Users/bsis0/Downloads/usa_housing.csv')

df.head()

,Price,Bedrooms,Bathrooms,SquareFeet,YearBuilt,GarageSpaces,LotSize,ZipCode,CrimeRate,SchoolRating
0,221958,1,1.9,4827,1979,2,1.45,82240,48.60,5
1,771155,2,2.0,1035,1987,2,1.75,74315,92.03,9
2,231932,1,3.0,2769,1982,1,1.46,79249,52.08,3
3,465838,3,3.3,2708,1907,3,1.62,80587,61.65,1
4,359178,4,3.4,1175,1994,2,0.74,20756,15.66,4


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         300 non-null    int64  
 1   Bedrooms      300 non-null    int64  
 2   Bathrooms     300 non-null    float64
 3   SquareFeet    300 non-null    int64  
 4   YearBuilt     300 non-null    int64  
 5   GarageSpaces  300 non-null    int64  
 6   LotSize       300 non-null    float64
 7   ZipCode       300 non-null    int64  
 8   CrimeRate     300 non-null    float64
 9   SchoolRating  300 non-null    int64  
dtypes: float64(3), int64(7)
memory usage: 23.6 KB


In [3]:
df2 = df[df['Bedrooms']>=3]

df2.info()

<class 'pandas.DataFrame'>
Index: 174 entries, 3 to 299
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         174 non-null    int64  
 1   Bedrooms      174 non-null    int64  
 2   Bathrooms     174 non-null    float64
 3   SquareFeet    174 non-null    int64  
 4   YearBuilt     174 non-null    int64  
 5   GarageSpaces  174 non-null    int64  
 6   LotSize       174 non-null    float64
 7   ZipCode       174 non-null    int64  
 8   CrimeRate     174 non-null    float64
 9   SchoolRating  174 non-null    int64  
dtypes: float64(3), int64(7)
memory usage: 15.0 KB


In [4]:
# X = df2[['CrimeRate']] #feature
# y = df2['Price'] #label

X = df2[['CrimeRate']]      # DataFrame 유지 ✅
y = df2[['Price']]          # DataFrame으로 통일 ← 여기만 수정!

# 아래 데이터 스케일링에서 밸류 에러 발생해서 y도 임의적으로 2차원 df로 변환함!!!

# 사이킷런에서 예전에는 넘파이값만 되었는데 이제는 위에처럼 데이터프레임과 시리즈로도 가능하다. (예제코드 - 머신러닝 기초 2장 2강 교안)
# X_sqft = df['SquareFeet'].values.reshape(-1, 1)  
# y_price = df['Price'].values.reshape(-1, 1)
# X_sqft.shape, y_price.shape

X,y

(     CrimeRate
 3        61.65
 4        15.66
 9        37.60
 10       55.01
 13       27.38
 ..         ...
 294      72.52
 295      32.30
 296      99.71
 298      65.61
 299      12.98
 
 [174 rows x 1 columns],
       Price
 3    465838
 4    359178
 9    737147
 10   621430
 13   275203
 ..      ...
 294  851102
 295  862002
 296  242483
 298  314835
 299  501896
 
 [174 rows x 1 columns])

In [5]:
# 데이터 쪼개기
from sklearn.model_selection import train_test_split 

# 전체를 6대 4로 분할 (train = 훈련용, temp = 임시)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=0.2, random_state=42)

# 40프로를 다시 5대 5로 분할 (val=검증용, test=테스트용)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, train_size=0.25, random_state=42)

In [6]:
print(f"전체 데이터:{len(X)}")
print(f"훈련 데이터:{len(X_train)}")
print(f"검증 데이터:{len(X_val)}")
print(f"테스트 데이터:{len(X_test)}")


전체 데이터:174
훈련 데이터:34
검증 데이터:35
테스트 데이터:105


In [7]:
# 데이터 표준화 스케일링 적용하기
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# 입력 데이터용 스케일러와 타깃 데이터용 스케일러 각각 독립적으로 생성
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X).flatten()
y = scaler_y.fit_transform(y)

print(X_scaled[:5], y[:5])
print(X_scaled.shape, y.shape)


[0.61756431 0.15544614 0.37590434 0.55084405 0.27321141] [[0.39656588]
 [0.27576013]
 [0.70385715]
 [0.57279322]
 [0.18064797]]
(174,) (174, 1)


In [8]:
# 학습률 설정 및 오차 기록 리스트 생성하기

# 모델 파라미터 초기화
weight = 0.0
bias = 0.0

# 하이퍼마라미터 초기값 설정
learning_rate = 0.05
epochs = 150

# 훈련 과정과 검증 과정에서 발생하는 손실의 변화를 기록할 빈 리스트 생성
tain_losses = []
valid_losses = []

print(f"""초기 가중치:{weight}\n설정한 학습률:{learning_rate}\n리스트 확인:{tain_losses},{valid_losses}""")

초기 가중치:0.0
설정한 학습률:0.05
리스트 확인:[],[]
